# Historial de la muestra

##### En la primer parte descargamos las features históricas para la muestra seleccionada del snap 28. En la segunda parte calculamos el historial de mergers.

In [1]:
import src.merger_tree_tools as mtt
import numpy as np
from astropy.table import Table
from astropy.cosmology import Planck15 as cosmo
import h5py
import pandas as pd
import polars as pl

In [2]:
# Ruta de los datos
path_lin ='data/' 

In [3]:
# Cargamos los datos de las galaxias centrales del snapshot 28 (z=0.1) para obtener los GalaxyID de cada una de ellas.

# Para la simu RefL0100N1504, el snapshot 28.
df  = pl.read_csv(path_lin+'data_centrales_snap28.csv')

# Proximamente dataframe sin AGN

In [4]:
# Primeras 10 gal de la muestra
df.head(10)

GalaxyID,GroupID,SnapNum,SubGroupNumber,Stars_Mass,BlackHoleMass,SF_Mass,SF_Oxygen,SF_Hydrogen,metalicidad
i64,i64,i64,i64,f64,f64,f64,f64,f64,f64
8646377,28000000000765,28,0,2.3954e10,2.332017e6,6.3366e9,0.013817,0.680822,9.103268
8650718,28000000000776,28,0,2.6976e10,1.6145332e7,3.6201e9,0.013211,0.682108,9.082946
8653534,28000000000784,28,0,4.0430e10,4039454.5,7.6696e9,0.016388,0.665448,9.187281
8657694,28000000000789,28,0,3.5193e10,1.6860834e7,1.5863e9,0.022328,0.621889,9.35102
8664336,28000000000811,28,0,3.6234e10,4919216.5,3.8502e9,0.01882,0.649435,9.257953
8669006,28000000000822,28,0,2.0129e10,2.9183e6,4.0235e9,0.013354,0.680497,9.088652
8672479,28000000000825,28,0,1.0131e10,1.7151e6,2.3637e9,0.01328,0.68166,9.085519
8675491,28000000000835,28,0,3.5932e10,1.4682328e7,4.8604e9,0.016661,0.660269,9.197848
8681166,28000000000856,28,0,3.1108e10,2.8666e6,4.9146e9,0.015695,0.667333,9.1673


In [5]:
# Longitud de la muestra
len(df)

2172

### Etapa de variables históricas

In [63]:
# Simulación y snapshot a analizar
simu='RefL0100N1504'
snap=28

# Lista de variables a descargar.

columns=[
          'GalaxyID', 'SnapNum','Redshift',
          # Propiedades de las estrellas
          'Stars_Mass','Stars_KineticEnergy','Stars_Spin_x','Stars_Spin_y','Stars_Spin_z',
          # Propiedades SF
          'SF_Mass','SF_Oxygen','SF_Hydrogen','SF_MassWeightedTemperature','SF_KineticEnergy',
          'SF_ThermalEnergy','SF_TotalEnergy','SF_Spin_x','SF_Spin_y','SF_Spin_z',
          # Propiedades de los Agujeros Negros
          'BlackHoleMass','BlackHoleMassAccretionRate'
        ]

In [64]:
long = len(df['GalaxyID'])

#long = 2

# Usuario y contraseña para conectarse a EAGLE DataBase
usr='cht015'
pwd='BH457tfj'

# Descargar merger tree completo de la galaxia deseada
# Nombre y alias de la tabla de la cual se quieren descargar datos
table='SubHalo'
table_alias='sub'


tracks = []

#Loop para todas las galaxias de la tabla
for p in np.arange(long):

    galid = df['GalaxyID'].to_numpy()[p]
 
    # Descargar todos los IDS necesarios de la galaxia deseada

    myIDs    = mtt.retrieve_ids(usr,pwd,simu,snap,galid)

    raw_tree = mtt.download_merger_tree(usr,pwd,simu,myIDs['GalaxyID'],myIDs['LastProgID'],
                                  table=table,table_alias=table_alias,columns=columns)
    
    # Aplicar condiciones a las galaxias del árbol, si es necesario
    mask = (
            (raw_tree['Stars_Mass']>0)&(raw_tree['SnapNum']>11) #& (raw_tree['Redshift']<2)
           )

    # Armar arbol sólo con galaxias seleccionadas según condiciones anteriores
    tree = {}
    for key in raw_tree.keys():
        tree[key]=raw_tree[key][mask]
        
    # Ordeno las galaxias según SnapNum creciente
    mask_order = (tree['SnapNum']).argsort()
    for key in tree.keys():
        tree[key] = tree[key][mask_order]

    # Armo un diccionario con solo la main branch del árbol
    main_branch = {}
    # Select galaxies in the main branch 
    mask_main = np.logical_and(tree['GalaxyID']>=myIDs['GalaxyID'],
                               tree['GalaxyID']<=myIDs['TopLeafID'])
    
    
    tmp = {
        key: tree[key][mask_main]
        for key in tree.keys()

    }
    tmp['RootGalaxyID'] = np.repeat(galid, len(tmp['GalaxyID']))

    tracks.append(pl.DataFrame(tmp))


In [65]:
# DataFrame final con la historia evolutiva de las galaxias centrales

tracks = pl.concat(tracks)

In [66]:
tracks.shape

(36854, 22)

In [67]:
# Guardamos el DataFrame con la historia evolutiva de las galaxias centrales
tracks.write_csv(path_lin+'tracks_centrales_snap28.csv')

In [68]:
tracks

GalaxyID,SnapNum,Redshift,Stars_Mass,Stars_KineticEnergy,Stars_Spin_x,Stars_Spin_y,Stars_Spin_z,SF_Mass,SF_Oxygen,SF_Hydrogen,SF_MassWeightedTemperature,SF_KineticEnergy,SF_ThermalEnergy,SF_TotalEnergy,SF_Spin_x,SF_Spin_y,SF_Spin_z,BlackHoleMass,BlackHoleMassAccretionRate,SubHaloGalaxyID,RootGalaxyID
i64,i32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,i64,i64
8646393,12,3.016505,1.81603728e8,8.9158e11,-82.791679,-55.607811,-12.126945,9.03976768e8,0.001334,0.747265,12797.511719,3.7069e12,2.2132e11,-2.7809e13,-47.896557,-75.066986,-49.205849,149624.953125,1.0145e-10,8646393,8646377
8646392,13,2.478413,4.04847584e8,1.6541e12,-27.682028,55.634319,90.331627,1.0362e9,0.002073,0.745057,11154.858398,3.3986e12,2.2240e11,-3.7311e13,-28.948641,177.069519,169.047363,149628.078125,2.2226e-9,8646392,8646377
8646391,14,2.237037,6.70437248e8,4.5140e12,-441.02179,-108.292862,311.478058,2.2858e9,0.002123,0.744817,11240.431641,1.2060e13,4.9555e11,-1.1021e14,-98.341614,193.810242,268.261658,149650.484375,7.4560e-9,8646391,8646377
8646390,15,2.01241,1.0876e9,6.8919e12,-590.810242,-142.639435,503.463623,2.9391e9,0.003144,0.741137,12233.349609,2.1326e13,6.9291e11,-1.5328e14,-486.672943,-134.645004,331.676636,149653.765625,7.1106e-9,8646390,8646377
8646389,16,1.736966,1.7818e9,8.2777e12,-443.328369,-363.820374,186.826813,3.5689e9,0.004233,0.736286,9807.323242,1.7867e13,6.7435e11,-2.1800e14,-560.238892,-350.904114,29.797092,298193.59375,2.2962e-8,8646389,8646377
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
21986365,24,0.365669,8.0355e11,1.3398e17,-8157.271484,-4815.602539,-7921.742188,1.5845e10,0.009007,0.71356,14842.010742,1.4543e15,4.3683e12,-1.8960e16,18130.810547,-38427.527344,-3099.356689,1.0961e9,0.00445,21986365,21986361
21986364,25,0.270901,9.1478e11,2.1425e17,-3263.486328,-13905.607422,-23574.283203,1.4140e10,0.009128,0.71065,15650.902344,2.7811e15,4.1468e12,-1.6325e16,13426.725586,-36767.789062,-18871.074219,1.1888e9,0.012013,21986364,21986361
21986363,26,0.18271,1.0332e12,1.9705e17,-2159.209961,-12064.918945,-21040.171875,1.0932e10,0.00931,0.711415,13835.003906,1.2363e15,2.7834e12,-1.4122e16,12981.541016,-47848.949219,-39117.738281,8.60549312e8,0.000002,21986363,21986361


In [32]:
lookbacktime = [0,1.35,2.32,3.24,4.12,5.22,5.98,6.69,7.35,7.96,8.86,9.49,10.05,10.54,10.87,11.17,11.68]
for i in range(len(lookbacktime)-1):
    print(f'Entre {lookbacktime[i]} y {lookbacktime[i+1]} Gyr hay {-lookbacktime[i]+lookbacktime[i+1]} Gyr')

Entre 0 y 1.35 Gyr hay 1.35 Gyr
Entre 1.35 y 2.32 Gyr hay 0.9699999999999998 Gyr
Entre 2.32 y 3.24 Gyr hay 0.9200000000000004 Gyr
Entre 3.24 y 4.12 Gyr hay 0.8799999999999999 Gyr
Entre 4.12 y 5.22 Gyr hay 1.0999999999999996 Gyr
Entre 5.22 y 5.98 Gyr hay 0.7600000000000007 Gyr
Entre 5.98 y 6.69 Gyr hay 0.71 Gyr
Entre 6.69 y 7.35 Gyr hay 0.6599999999999993 Gyr
Entre 7.35 y 7.96 Gyr hay 0.6100000000000003 Gyr
Entre 7.96 y 8.86 Gyr hay 0.8999999999999995 Gyr
Entre 8.86 y 9.49 Gyr hay 0.6300000000000008 Gyr
Entre 9.49 y 10.05 Gyr hay 0.5600000000000005 Gyr
Entre 10.05 y 10.54 Gyr hay 0.48999999999999844 Gyr
Entre 10.54 y 10.87 Gyr hay 0.33000000000000007 Gyr
Entre 10.87 y 11.17 Gyr hay 0.3000000000000007 Gyr
Entre 11.17 y 11.68 Gyr hay 0.5099999999999998 Gyr


### Analizamos el historial de merger de las galxias.

In [6]:
# Simulación y snapshot a analizar
simu='RefL0100N1504'
snap=28

In [7]:
# Va a contener el historial de fusiones
rows = []

In [8]:
long = len(df['GalaxyID'])

#long = 5

# Usuario y contraseña para conectarse a EAGLE DataBase
usr='cht015'
pwd='BH457tfj'

# Descargar merger tree completo de la galaxia deseada
# Nombre y alias de la tabla de la cual se quieren descargar datos
table='SubHalo'
table_alias='sub'

# Variables que se quiere descargar. OJO!! Asegurarse que estas variables
# estén en la tabla deseada.
columns=[
          'GalaxyID','LastProgID','TopLeafID','DescendantID',
          'SnapNum','Redshift','Stars_Mass','SF_Mass','GroupID'
        ]

#Loop para todas las galaxias de la tabla
for p in np.arange(long):

    galid = df['GalaxyID'].to_numpy()[p]
 
    # Descargar todos los IDS necesarios de la galaxia deseada

    myIDs    = mtt.retrieve_ids(usr,pwd,simu,snap,galid)

    raw_tree = mtt.download_merger_tree(usr,pwd,simu,myIDs['GalaxyID'],myIDs['LastProgID'],
                                  table=table,table_alias=table_alias,columns=columns)
    
    # Aplicar condiciones a las galaxias del árbol, si es necesario
    mask = (
            (raw_tree['Stars_Mass']>0)
           )

    # Armar arbol sólo con galaxias seleccionadas según condiciones anteriores
    tree = {}
    for key in raw_tree.keys():
        tree[key]=raw_tree[key][mask]
        
    # Ordeno las galaxias según SnapNum creciente
    mask_order = (tree['SnapNum']).argsort()
    for key in tree.keys():
        tree[key] = tree[key][mask_order]

    # Armo un diccionario con solo la main branch del árbol
    main_branch = {}
    # Select galaxies in the main branch 
    mask_main = np.logical_and(tree['GalaxyID']>=myIDs['GalaxyID'],
                               tree['GalaxyID']<=myIDs['TopLeafID']) 

    for key in tree.keys():
        main_branch[key] = tree[key][mask_main]

        
    xplot_main = main_branch['SnapNum']
    
    # Calculo level of merger
    level_merger=[1]       # Inicializo con un 1, porque la primer galaxia del main branch 
                           # no viene de ninguna fusión...
    
    for k in range(np.size(xplot_main)-1):
        m1=main_branch['Stars_Mass'][k]+main_branch['SF_Mass'][k]
        mask=(tree['DescendantID']==main_branch['GalaxyID'][k+1]) & (tree['SnapNum']!=28) 
        # La condición de snapnum distinto a 28 es porque galaxias a z=0 no tienen descendiente,
        # y se les asigna como DescendantID su propio GalaxyID
        m2=np.sum(tree['Stars_Mass'][mask])+np.sum(tree['SF_Mass'][mask])
        level=m2/m1
        level_merger=np.append(level_merger,level)
        
    for snap, Lm_value in zip(main_branch['SnapNum'], level_merger):
        rows.append({
            "GalaxyID": galid,
            "Snapshot": int(snap),
            "Lm": float(Lm_value)
        })  
        

    
    print('Porcentaje:',round(float((p+1)/long),3),end='\r')


In [9]:
DATA_merge = pl.DataFrame(rows)
DATA_merge

GalaxyID,Snapshot,Lm
i64,i64,f64
8646377,3,1.0
8646377,4,1.0
8646377,5,1.0
8646377,6,1.0
8646377,7,1.0
…,…,…
21986361,24,1.001543
21986361,25,1.000458
21986361,26,1.041736


In [ ]:
#mask = (pl.col('Snapshot')>11) & (pl.col('Lm')>1.1)
#DATA_merge.filter(mask)

GalaxyID,Snapshot,Lm
i64,i64,f64
8646377,13,1.208551
8650718,13,1.196043
8664336,12,1.207541
8664336,19,1.122106
8664336,23,1.115946
8664336,28,1.119801


In [ ]:
# Guardamos el DataFrame con el historial de fusiones de las galaxias centrales
DATA_merge.write_csv(path_lin+'historial_merger_centrales_snap28.csv')

In [42]:
# Chusmeo la cantidad de fusiones que hay por snapshot, Lm>1.1 y Lm<2.0
mask = (pl.col('Snapshot')==21) & (pl.col('Lm')<1.1) & (pl.col('Lm')<2.1)
DATA_merge.filter(mask)

GalaxyID,Snapshot,Lm
i64,i64,f64
8646377,21,1.037294
8650718,21,1.00012
8653534,21,1.016903
8657694,21,1.017706
8664336,21,1.012252
…,…,…
21379521,21,1.032657
21573586,21,1.017296
21730535,21,1.017061


In [14]:
table_a.to_csv(path_or_buf= path_lin+'/bin_proyecto_altas_masas_mayores_a_10_NoAGNL0050N0752_snap_28.dat',index=False)

* Snapshot 19 = Redshift 1
* Snapshot 28 = Redshift 0

## Pruebas fraction of merger

In [3]:
DATA_merge  = pd.read_csv(path_lin+'/Actividad_8(merger_history_tesis).dat')

In [20]:
mask = (DATA_merge['Lm_snap6'] == 0)
DATA_merge[mask]

,GalaxyID_test,GroupID_test,Lm_snap1,Lm_snap2,Lm_snap3,Lm_snap4,Lm_snap5,Lm_snap6,Lm_snap7,Lm_snap8,...,Lm_snap19,Lm_snap20,Lm_snap21,Lm_snap22,Lm_snap23,Lm_snap24,Lm_snap25,Lm_snap26,Lm_snap27,Lm_snap28
7,2727611,28000000000002,NaN,NaN,1.0,NaN,NaN,0.0,1.891800,1.000000,...,1.018315,1.033089,1.068498,1.000000,NaN,NaN,NaN,0.00000,NaN,NaN
14,4470669,28000000000004,NaN,NaN,1.0,1.0,NaN,0.0,1.000000,1.000000,...,1.023634,1.000000,1.027689,1.029271,1.015195,1.006829,1.000000,1.00000,1.000000,NaN
18,10507082,28000000002364,NaN,NaN,1.0,NaN,NaN,0.0,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000
24,10352660,28000000002189,NaN,NaN,1.0,1.0,NaN,0.0,1.000000,1.000000,...,1.000000,1.474600,1.333145,1.000000,1.000000,1.007035,1.000000,1.00000,1.000000,1.000000
34,21793,28000000000000,NaN,1.0,NaN,NaN,NaN,0.0,2.658798,1.000000,...,1.043703,1.000000,1.004681,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3595,8667711,28000000000817,NaN,1.0,NaN,NaN,NaN,0.0,1.000000,1.000000,...,1.047138,1.029600,1.006176,1.000000,1.037501,1.012499,1.000626,1.00000,1.000000,1.000000
3602,18385552,28000000001003,NaN,1.0,NaN,NaN,NaN,0.0,1.000000,1.000000,...,1.000000,1.010950,1.016656,1.026810,1.010499,1.007426,1.011448,1.00701,1.009407,1.017955
3614,2979098,28000000001336,NaN,NaN,NaN,1.0,NaN,0.0,1.000000,1.000000,...,1.000000,1.000000,1.006409,1.000000,1.000000,1.023178,1.007344,1.00000,1.033620,1.016327
3622,9733375,28000000001609,NaN,NaN,1.0,NaN,NaN,0.0,1.000000,1.000000,...,1.042202,1.000000,1.000000,1.000000,1.000000,1.003019,1.000000,1.00000,1.000000,1.022765


In [ ]:
# Fin